In [1]:
# import library
import time
import random
import requests

from bs4 import BeautifulSoup
import pandas as pd
import re

In [2]:
debug_mode = False

In [3]:
if debug_mode:
    df_urls = pd.read_csv('urls.csv')

    baseurl = df_urls.baseurl[0]
    filterurl = df_urls.filterurl[0]

In [11]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36',
           'Accept-Language': 'de-CH'}

ip = '150.107.140.238:3128'
proxies = {'http': f'http://{ip}', 'https': f'http://{ip}'}

### Step 1: Request Listings and Export to CSV

In [ ]:
def request_listings(baseurl, filterurl, pages):
    raw_listings = []

    for n in range(1, pages + 1):
        url = f'{baseurl}hotel-page-{n}{filterurl}'
        raw_listings.append(requests.get(url, headers=headers, proxies=proxies).text)
        time.sleep(max(random.gauss(3, 1), 2))
    
    return raw_listings

In [ ]:
if debug_mode:
    raw_listings = request_listings(baseurl, filterurl, 367)

In [13]:
# export raw listings to csv
if debug_mode:
    pd.DataFrame({'raw_listing': raw_listings}).to_csv('raw_listings.csv', index=False)

### Step 2: Extract Unique Partial Link, Companies and Ratings from Raw Listings

In [ ]:
# load raw listings from csv
if debug_mode:
    raw_listings = pd.read_csv('raw_listings.csv').raw_listing.to_list()

In [ ]:
def extract_links(raw_listings):
    links = []

    for l in raw_listings:
        soup = BeautifulSoup(l, 'lxml').findAll('a', 'Card small')
        
        for e in soup:
            links.append(e.get('href').split('/')[-1])

    return links

In [ ]:
if debug_mode:
    links = extract_links(raw_listings)

In [ ]:
# export unique partial links to csv
if debug_mode:
    pd.DataFrame({'links': links}).to_csv('links.csv', index=False)

In [ ]:
def extract_companies(raw_listings):
    companies = []

    for l in raw_listings:
        soup = BeautifulSoup(l, 'lxml').findAll('a', 'Card small')
        
        for e in soup:
            companies.append(e.text.split('\n')[7].strip())

    return companies

In [ ]:
if debug_mode:
    companies = extract_companies(raw_listings)

In [ ]:
def extract_ratings(raw_listings):
    ratings = []

    for l in raw_listings:
        soup = BeautifulSoup(l, 'lxml').findAll('a', 'Card small')
        
        for e in soup:
            if (found := str(e.find('span', class_=re.compile(r'Stars*')))) != 'None':
                if (found := found.split('"')[1][18:25]) != 'lodge':
                    ratings.append(found)
                else:
                    ratings.append('0')
            else:
                ratings.append('0')
                
    return ratings

In [ ]:
if debug_mode:
    ratings = extract_ratings(raw_listings)

### Step 3: Get Contact

In [ ]:
# load extracted email page, unique partial links from csv
if debug_mode:
    links = pd.read_csv('links.csv').links.to_list()

In [ ]:
# build full email page link using base url + unique partial link
if debug_mode:
    links = [f'{baseurl}{link}' for link in links]

In [ ]:
def request_emails(links):
    email_pages = []

    for link in links:
        email_pages.append(requests.get(link, headers=headers, proxies=proxies).text)
        time.sleep(max(random.gauss(3, 1), 2))

    return email_pages

In [ ]:
if debug_mode:
    raw_emails = request_emails(links)

In [11]:
# save raw, not yet extracted emails to csv
if debug_mode:
    pd.DataFrame({'raw_emails': raw_emails}).to_csv('raw_emails.csv', index=False)

In [5]:
# load raw, not yet extracted emails
if debug_mode:
    raw_emails = pd.read_csv('raw_emails.csv').raw_emails.to_list()

In [ ]:
def decoder(string):
    result = bytearray(string, 'ascii')
    
    for i, c in enumerate(result):
        if c == 97:
            result[i] = 122
        else:
            result[i] -= 1
            
    return result.decode('ascii')

In [ ]:
def extract_emails(raw_emails):
    emails = []

    for i, email in enumerate(raw_emails):
        soup = BeautifulSoup(email, 'lxml').findAll('a', "Button secondary sidebar")
        
        if (email_encoded := soup[1].get('data-mailto-token')) or (email_encoded := soup[0].get('data-mailto-token')):
            emails.append(decoder(email_encoded[7:]))
        else:
            emails.append('0')
            
    return emails

In [ ]:
if debug_mode:
    sample = 100
    emails = extract_emails(raw_emails[:sample])

### Step 4: Export All Extracted Data to CSV

In [ ]:
# save final data to csv
if debug_mode:
    pd.DataFrame({'hotel': companies, 'rating': ratings, 'email': emails, 'link': links}).to_csv('hotels_final.csv', index=False)

In [ ]:
# load final data
if debug_mode:
    df_final = pd.read_csv('hotels_final.csv', encoding="ISO-8859-1")

### DEBUG CODE

In [ ]:
if debug_mode:
    results = []
    rounds = 30
    sample = 100

    for _ in range(rounds):
        start = time.perf_counter()
        
        extract_emails(raw_emails[:sample])
        
        end = time.perf_counter()
        results.append(end-start)

    print(sum(results)/rounds)